In [37]:
# Import libraries - numpy, pandas, matplotlib, seaborn, and scikit-learn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split

# Make it so plots display in the notebook
%matplotlib inline

In [38]:
# Read and view the data - 2026 Lead-Scoring data. Call it leads.
leads = pd.read_csv("https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv")
leads.tail()
# 5000 data points
# Target variable is converted - 1 if a lead signed up for the platform, and 0 if not

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
4995,organic_search,education,employed,asia,45834.0,2,2,0.37,1
4996,NaN,technology,employed,north_america,84831.0,2,5,0.51,0
4997,organic_search,finance,employed,asia,53636.0,4,6,0.75,1
4998,referral,technology,employed,africa,69430.0,4,8,0.79,1
4999,referral,technology,employed,europe,81355.0,6,11,0.99,1


In [39]:
# Data Preparation
# Check if the missing values are presented in the features.
# If there are missing values:
# For categorical features, replace them with 'NA'
# For numerical features, replace with with 0.0

# Now check for missing values
leads.isnull().sum()
# lead_source, industry, employment_status, location, annual_income, and lead_score all have missing values.

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [40]:
# Check the variables to determine if they are categorical or numeric
pd.DataFrame([leads.dtypes, leads.nunique()], index=["Data Type", "Unique Values"])

# From the results of this and the tail() examination earlier, can determine the following:
# categorical variables: lead_source, industry, employment_status, location
# numeric variables: annual_income, lead_score

# None of the categorical variables are binary, aside from the target

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
Data Type,str,str,str,str,float64,int64,int64,float64,int64
Unique Values,5,6,4,5,4313,7,14,99,2


In [41]:
# Save these variables to fill to lists
categorical_vars = ['lead_source', 'industry', 'employment_status', 'location']
numeric_vars = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


# For each categorical variable, replace missing values with 'NA' as instructed
leads[categorical_vars] = leads[categorical_vars].fillna('NA')

# For each numeric variable, replace missing values with 0.0 as instructed
leads[numeric_vars] = leads[numeric_vars].fillna(0.0)

In [42]:
# Question 1
# What is the most frequent observation (mode) for the column industry?

# Can do this by calling describe() on the column in question
leads.industry.describe()
# The most frequent industry is technology

count           5000
unique             7
top       technology
freq            1173
Name: industry, dtype: object

In [43]:
# Question 2
# Create the correlation matrix for the numerical features of your dataset. 
# In a correlation matrix, you compute the correlation coefficient between every pair of features.
# What are the two features that have the biggest correlation?
# interaction_count and lead_score
# number_of_courses_viewed and lead_score
# number_of_courses_viewed and interaction_count
# annual_income and interaction_count

# The corr() function returns the correlation matrix. Only pass the numeric vars into the function.
leads[numeric_vars].corr()
# interaction_count and lead_score has a correlation of appx 0.92, a very strong positive relationship
# number_of_courses_viewed and lead_score has a correlation of appx 0.76, a strong positive relationship
# number_of_courses_viewed and interaction_count has a correlation of appx 0.72, a strong positive relationship
# annual_income and interaction_count has a correlation of appx 0.12, a weak positive relationship

# Of these, interaction_count and lead_score have the biggest correlation at appx 0.92.

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [44]:
# Split the data using Scikit-Learn's train_test_split function, as instructed
df_full_train, df_test = train_test_split(leads, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

# Extract X_train, y_train, X_val, y_val, X_test, and y_test
X_train = df_train.loc[:, df_train.columns != 'converted']
y_train = df_train.converted.values
X_val = df_val.loc[:, df_val.columns != 'converted']
y_val = df_val.converted.values
X_test = df_test.loc[:, df_test.columns != 'converted']
y_test = df_test.converted.values

In [45]:
# Let's view the conversion rate of the full training set
df_full_train.converted.mean()
# See that appx 57% convert and therefore appx 43% don't. Conversion rate is 57.075%.

np.float64(0.57075)

In [46]:
# Let's examine how conversion date is affected by the various categorical variables.
converted_info = pd.DataFrame()
for var in categorical_vars:
    print(df_full_train.groupby(var).converted.mean() / df_full_train.converted.mean())
# Leads who found the service through referral were 34% more likely to convert than average.
# Leads who found the service by organic search were 19% more likely to convert than average.
# Leads who were employed were 15% more likely to convert than average.
# Leads who found the service through events were 28% less likely to convert than average.
# Leads who found the service through paid ads were 25% less likely to convert than average.
# Leads who were students were 26% less likely to convert than average.
# Leads who were unemployed were 24% less likely to convert than average.
# All of these make sense.
# Industry and location seem to matter less than lead_source and employment_status, but they all seem to matter somewhat.

lead_source
NA                1.069066
events            0.720614
organic_search    1.187174
paid_ads          0.749815
referral          1.338316
social_media      0.960073
Name: converted, dtype: float64
industry
NA               1.029573
education        0.876040
finance          1.074730
healthcare       0.980111
manufacturing    0.961838
retail           0.970200
technology       1.070716
Name: converted, dtype: float64
employment_status
NA               1.016639
employed         1.150170
self_employed    0.960623
student          0.735476
unemployed       0.763157
Name: converted, dtype: float64
location
NA               1.057469
africa           0.956906
asia             1.025507
europe           1.024654
north_america    1.010816
south_america    0.920633
Name: converted, dtype: float64


In [47]:
# Question 3
# Calculate the mutual information score between converted and the other categorical variables in the dataset. Use the training set only.
# Round the scores to 2 decimals using round(score, 2).
# Which of these variables has the biggest mutual information score:
# industry, location, lead_source, or employment_status?

# For the training set only, calculate the mutual information scores between converted and the other categorical variables.
# Use mutual_info_score from sklearn.metrics

from sklearn.metrics import mutual_info_score
score = df_train[categorical_vars].apply(mutual_info_score, labels_pred=df_train.converted, axis=0)
score
# By mutual importance,
# lead_source is the most important
# employment_status is also important
# industry and location don't seem to be important
# This is in line with the conversion rate analysis we did earlier.

lead_source          0.034346
industry             0.001995
employment_status    0.018966
location             0.001043
dtype: float64

In [48]:
# Compare method to one shown in class
def mutual_info_converted_score(series):
    return mutual_info_score(series, df_train.converted)

df_train[categorical_vars].apply(mutual_info_converted_score)
# Same result!

lead_source          0.034346
industry             0.001995
employment_status    0.018966
location             0.001043
dtype: float64

In [49]:
# Round the scores to 2 decimals using round(score, 2).
round(score, 2)
# Once again, it's clear lead_source has the biggest mutual information score.

lead_source          0.03
industry             0.00
employment_status    0.02
location             0.00
dtype: float64

In [50]:
# Question 4
# Now let's train a logistic regression.
# Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
# Fit the model on the training dataset.
# To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
# model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
# Calculate the accuracy on the validation dataset and round it to 2 decimal digits.
# What accuracy did you get?

# First need to one-hot encode the categorical variables. Use pd.get_dummies().
from sklearn.preprocessing import OneHotEncoder
X_train = pd.get_dummies(X_train, columns=categorical_vars, drop_first=True)
X_val = pd.get_dummies(X_val, columns=categorical_vars, drop_first=True)
X_test = pd.get_dummies(X_test, columns=categorical_vars, drop_first=True)

# Ensure all three sets have the same columns
X_train.columns == X_val.columns, X_val.columns == X_test.columns
# All good!

(array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True]),
 array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True]))

In [51]:
# Question 4
# Train a logistic regression model as instructed.
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# Calculate the accuracy on the validation dataset and round it to 2 decimal digits.
y_pred_probs = model.predict_proba(X_val)[:, 1]
y_pred = (y_pred_probs >= 0.5).astype(int)
accuracy_four = (y_val == y_pred).mean()
round(accuracy_four, 2)
# Accuracy is appx 64%. Model's prediction on validation data is appx 64% accurate. Not great.

np.float64(0.64)

In [52]:
# Question 5
# Let's find the least useful feature using the feature elimination technique.
# Train a model using the same features and parameters as in Q4 (without rounding).
# Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
# For each feature, calculate the difference between the original accuracy and the accuracy without the feature.
# Which of following features has the smallest difference? 'lead_source', 'number_of_courses_viewed', or 'interaction_count'

# Luckily we saved the accuracy from Q4 without rounding to accuracy_four so we don't need to train it again.
# Start by creating accuracies_five to store the accuracies from each new model trained.
# The length of this array is the number of features, len(X_train.columns)
accuracies_five = np.zeros(len(X_train.columns))

# For each column, train a model without it using the same process as in question 4.
for i in range(len(X_train.columns)):
    # Set X_train_five to X_train with the ith column excluded. Set X_val_five to X_val with the same column excluded. 
    X_train_five = X_train.loc[:, X_train.columns != X_train.columns[i]]
    X_val_five = X_val.loc[:, X_val.columns != X_train.columns[i]]
    # Train the model on X_train_five
    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train_five, y_train)
    # Calculate the accuracy on the modified validation dataset and save it to accuracies_five
    y_pred_probs = model.predict_proba(X_val_five)[:, 1]
    y_pred = (y_pred_probs >= 0.5).astype(int)
    accuracies_five[i] = (y_val == y_pred).mean()

# Return the differences between the accuracy from Question 4 and the new accuracies_five
differences = pd.DataFrame({
    'column removed': X_train.columns,
    'difference': accuracies_five - accuracy_four
})
differences
# Removing number_of_courses_viewed decreased the accuracy by 0.002.
# Removing interaction_count decreased the accuracy by 0.044.

# lead_source was One-Hot encoded into five variables. Can't say for sure what the overall difference is.
# Need to repeat when excluding all five lead_source variables.

,column removed,difference
0,annual_income,0.080
1,number_of_courses_viewed,-0.002
2,interaction_count,-0.044
3,lead_score,-0.001
4,lead_source_events,0.002
5,lead_source_organic_search,0.001
6,lead_source_paid_ads,-0.002
7,lead_source_referral,0.000
8,lead_source_social_media,0.001
9,industry_education,0.001


In [53]:
# Question 5

# Set X_train_five to X_train with the lead_source columns excluded. Set X_val_five to X_val with the same columns excluded. 
X_train_five = X_train[[col for col in X_train.columns if not col.startswith('lead_source')]]
X_val_five = X_val[[col for col in X_train.columns if not col.startswith('lead_source')]]

# Train the model on X_train_five
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train_five, y_train)

# Calculate the accuracy on the modified validation dataset and save it to accuracy_lead_source
y_pred_probs = model.predict_proba(X_val_five)[:, 1]
y_pred = (y_pred_probs >= 0.5).astype(int)
accuracy_lead_source = (y_val == y_pred).mean()

# Return the difference between this accuracy and the accuracy from Question 4
accuracy_lead_source - accuracy_four
# Removing lead_source decreased the accuracy by 0.003. Not a big difference.
# However, number_of_courses_viewed had an even smaller difference of 0.002.

np.float64(-0.0030000000000000027)

In [54]:
# Question 6
# Now let's train a regularized logistic regression.
# Let's try the following values of the parameter C: [0.000001, 0.00001, 0.0001, 0.001].
# Train models using all the features as in Q4.
# Calculate the accuracy on the validation dataset and round it to 3 decimal digits.
# Which of these C leads to the best accuracy on the validation set?

# Start by creating cs that stores the values of parameter C to test
cs = [0.000001, 0.00001, 0.0001, 0.001]

# Also create accuracies_six, which will store the accuracies from each iteration
accuracies_six = np.zeros(4)
accuracies_six

# For each value of C, repeat the process from question 4 and add the resulting accuracy to accuracies_six
for i in range(len(cs)):
    # Train the model with the new value of the parameter
    model = LogisticRegression(solver='liblinear', C=cs[i], max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    # Calculate the accuracy on the validation dataset and round it to 3 decimal digits.
    y_pred_probs = model.predict_proba(X_val)[:, 1]
    y_pred = (y_pred_probs >= 0.5).astype(int)
    accuracies_six[i] = round((y_val == y_pred).mean(), 3)

# Report the accuracies
accuracies_six
# Both C = 0.000001 and C = 0.00001 produce the same accuracy of 59.8% after rounding.
# As instructed, the best value of C is the smaller, which is 0.000001

array([0.598, 0.598, 0.613, 0.645])